# 03 - CLIP image embeddings for zero-shot crash scoring (Kaggle GPU)

**Why:** CLIP was trained on image + text pairs, so its image vectors live in the same space as text. Later we can ask
"is this frame closer to the text *'a car crash'* or to *'normal driving'*?" without using any train label at all.
A score like that cannot learn train-set quirks, which is exactly our problem (CV 0.94 but public LB 0.60).

**This notebook only saves image embeddings** (the slow GPU part). Scoring with text prompts happens later on the laptop, where we can try many prompts for free.

**Details:** every 2nd frame (15 per clip). CLIP wants square 224x224 images but our frames are 384x224, so each frame is cut into
a **left** and a **right** square (they overlap in the middle) and both are embedded. A crash on the left or right edge is never cropped away.
**Output:** `clip_l14_train.npy` (3000, 15, 2, 768) and `clip_l14_test.npy` (1750, 15, 2, 768), L2-normalised, float16.
**Settings on Kaggle:** GPU T4, Internet On (model download), competition `cooked-or-not` attached. Local smoke test: env var `SMOKE=50`.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU found)")

## 1. Find data, choose output folder
`SMOKE` keeps only the first N clips per split for a fast local test.

In [ ]:
found = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True) + glob.glob("../../data/train_labels.csv")
assert found, "train_labels.csv not found - attach the competition / check the path"
ROOT = os.path.dirname(found[0])

SMOKE = int(os.environ.get("SMOKE", 0)) or None
on_kaggle = os.path.exists("/kaggle/working")
OUT = "/kaggle/working/features" if on_kaggle else ("../../artifacts/clip_smoke" if SMOKE else "../../artifacts/features")
os.makedirs(OUT, exist_ok=True)

train_ids = pd.read_csv(f"{ROOT}/train_labels.csv")["clip_id"].tolist()[:SMOKE]       # same order as the csv files
test_ids = pd.read_csv(f"{ROOT}/sample_submission.csv")["clip_id"].tolist()[:SMOKE]
print(ROOT, "| train clips:", len(train_ids), "| test clips:", len(test_ids))

## 2. Load CLIP ViT-L/14
Only used for inference (`.eval()`), 16-bit numbers on the GPU for speed. CLIP expects its own normalisation numbers (different from DINOv2).

In [ ]:
MODEL = "openai/clip-vit-large-patch14"
model = CLIPModel.from_pretrained(MODEL).eval().to(device)
if device == "cuda":
    model = model.half()

mean = torch.tensor([0.4815, 0.4578, 0.4082], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.2686, 0.2613, 0.2758], device=device).view(1, 3, 1, 1)
FRAMES = list(range(0, 30, 2))          # frames 0, 2, 4, ... 28 -> 15 frames per clip

## 3. Read one clip
Returns a tensor (15 frames, 2 views, 3, 224, 224): for each chosen frame, the left square and the right square.

In [ ]:
class ClipDataset(torch.utils.data.Dataset):
    def __init__(self, split, ids):
        self.split, self.ids = split, ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        folder = f"{ROOT}/{self.split}/{self.ids[i]}"
        views = []
        for j in FRAMES:
            img = np.asarray(Image.open(f"{folder}/frame_{j:03d}.jpg").convert("RGB"))   # (224, 384, 3)
            left, right = img[:, :224], img[:, -224:]                                    # two 224x224 squares
            views.append(np.stack([left, right]))
        views = np.stack(views)                                                          # (15, 2, 224, 224, 3)
        return torch.from_numpy(views).permute(0, 1, 4, 2, 3)                            # (15, 2, 3, 224, 224)

## 4. Embed
All views of a few clips are flattened into one batch, normalised, passed through CLIP's image tower, and **L2-normalised**
(length 1), so that a dot product with a text vector later equals cosine similarity.

In [ ]:
@torch.no_grad()
def embed_split(split, ids, batch_clips=2):
    loader = torch.utils.data.DataLoader(ClipDataset(split, ids), batch_size=batch_clips, shuffle=False,
                                         num_workers=0 if os.name == "nt" else 2)
    results = []
    for clips in tqdm(loader, desc=split):
        n = clips.shape[0]
        x = clips.flatten(0, 2).to(device).float() / 255                 # (n*15*2, 3, 224, 224) in 0..1
        x = (x - mean) / std
        if device == "cuda":
            x = x.half()
        pooled = model.vision_model(pixel_values=x).pooler_output        # image tower output (works in old and new transformers)
        f = model.visual_projection(pooled)                              # project into the shared image/text space (n*30, 768)
        f = f / f.norm(dim=-1, keepdim=True)
        results.append(f.view(n, len(FRAMES), 2, -1).float().cpu().numpy().astype(np.float16))
    return np.concatenate(results)                                       # (clips, 15, 2, 768)

## 5. Run and save
Full run shapes: (3000, 15, 2, 768) and (1750, 15, 2, 768), no NaNs.

In [ ]:
t0 = time.time()
train_feats = embed_split("train", train_ids)
test_feats = embed_split("test", test_ids)
np.save(f"{OUT}/clip_l14_train.npy", train_feats)
np.save(f"{OUT}/clip_l14_test.npy", test_feats)
pd.Series(train_ids, name="clip_id").to_csv(f"{OUT}/clip_train_ids.csv", index=False)
pd.Series(test_ids, name="clip_id").to_csv(f"{OUT}/clip_test_ids.csv", index=False)
print(train_feats.shape, test_feats.shape, f"{time.time() - t0:.0f}s", "NaNs:", np.isnan(train_feats).sum(), np.isnan(test_feats).sum())

## Next (local)
`kaggle kernels output aryanbhendarkar/cooked-or-not-03-clip-zeroshot -p artifacts/clip_dl`, then `notebooks/04_clip_zeroshot.ipynb`: text prompts -> per-frame crash score -> clip score -> AUC on train.